In [36]:
import re

# 1. U.S. ZIP codes
zip_pattern = re.compile(r"\b\d{5}(?:[-\s]\d{4})?\b")
zip_tests = ["12345", "12345-6789", "12345 6789", "123456", "abc12345"]
print("ZIP:", [m.group() for t in zip_tests for m in zip_pattern.finditer(t)])

# 2. Words not starting with a capital letter
word_pattern = re.compile(r"\b(?![A-Z])[A-Za-z]+(?:['’-][A-Za-z]+)*\b")
word_tests = ["dog", "Cat", "don’t", "state-of-the-art", "Hello"]
print("Non-capitalized words:", [m.group() for t in word_tests for m in word_pattern.finditer(t)])

# 3. Numbers (signs, commas, decimals, scientific notation)
num_pattern = re.compile(r"[+-]?(?:\d{1,3}(?:,\d{3})*|\d+)(?:\.\d+)?(?:[eE][+-]?\d+)?")
num_tests = ["123", "+123.45", "-1,234", "1.23e-4", "12,345,678.90e+10"]
print("Numbers:", [m.group() for t in num_tests for m in num_pattern.finditer(t)])

# 4. Email spelling variants
email_pattern = re.compile(r"(?i)\be[-\s–]?mail\b")
email_tests = ["email", "E-mail", "e mail", "E–mail", "Mail"]
print("Email variants:", [m.group() for t in email_tests for m in email_pattern.finditer(t)])

# 5. Interjection go/goo/gooo... with optional punctuation
go_pattern = re.compile(r"\bgo+[\!\.\,\?]?\b")
go_tests = ["go", "goo", "gooo!", "go?", "gone", "gooo,"]
print("Go variants:", [m.group() for t in go_tests for m in go_pattern.finditer(t)])

ZIP: ['12345', '12345-6789', '12345 6789']
Non-capitalized words: ['dog', 'don’t', 'state-of-the-art']
Numbers: ['123', '+123.45', '-1,234', '1.23e-4', '12,345,678.90e+10']
Email variants: ['email', 'E-mail', 'e mail', 'E–mail']
Go variants: ['go', 'goo', 'gooo', 'go', 'gooo']


In [35]:
import re
question_pattern = re.compile(r"\?[)\"'\]\s]*$")
question_tests = [
    "Is this working?",
    "What time is it?\") ",
    "Really?'",
    "No way!",
]
print("Questions:", [t for t in question_tests if question_pattern.search(t)])

Questions: ['Is this working?', 'What time is it?") ', "Really?'"]


In [37]:
from collections import Counter

def pair_counts(corpus):
    """Count adjacent token pairs, weighted by word frequency."""
    counts = Counter()
    for tokens, frequency in corpus.items():
        for pair in zip(tokens, tokens[1:]):
            counts[pair] += frequency
    return counts


def merge_pair(tokens, pair):
    """Merge nonoverlapping occurrences from left to right."""
    result, index = [], 0
    while index < len(tokens):
        if tuple(tokens[index:index + 2]) == pair:
            result.append("".join(pair))
            index += 2
        else:
            result.append(tokens[index])
            index += 1
    return tuple(result)


def train_bpe(words, maximum_merges):
    """Keep base characters and every learned token in the vocabulary."""
    corpus = Counter({tuple(w) + ("_",): n
                      for w, n in Counter(words).items()})
    vocabulary = {t for word in corpus for t in word}
    history = []
    for step in range(1, maximum_merges + 1):
        counts = pair_counts(corpus)
        if not counts:
            break
        # Highest frequency wins; ties use alphabetical pair order.
        pair = min(counts, key=lambda p: (-counts[p], p))
        updated = Counter()
        for tokens, frequency in corpus.items():
            updated[merge_pair(tokens, pair)] += frequency
        corpus = updated
        vocabulary.add("".join(pair))
        history.append({
            "step": step, "pair": pair, "count": counts[pair],
            "token": "".join(pair), "vocab_size": len(vocabulary)
        })
    return history, vocabulary, corpus


def encode_bpe(word, history):
    """Apply learned merges in training order to a new word."""
    tokens = tuple(word) + ("_",)
    for item in history:
        tokens = merge_pair(tokens, tuple(item["pair"]))
    return list(tokens)


def print_history(history):
    for item in history:
        a, b = item["pair"]
        print(f"{item['step']:2}: ({a}, {b}) -> "
              f"{item['token']}; count={item['count']}; "
              f"vocabulary={item['vocab_size']}")
TOY = ("low low low low low lowest lowest "
       "newer newer newer newer newer newer "
       "wider wider wider new new")
TOY_TARGETS = ["new", "newer", "lowest", "widest", "newestest"]
toy_history, toy_vocabulary, _ = train_bpe(TOY.split(), 20)
print_history(toy_history)
for word in TOY_TARGETS:
    print(word, encode_bpe(word, toy_history))

 1: (e, r) -> er; count=9; vocabulary=12
 2: (er, _) -> er_; count=9; vocabulary=13
 3: (e, w) -> ew; count=8; vocabulary=14
 4: (n, ew) -> new; count=8; vocabulary=15
 5: (l, o) -> lo; count=7; vocabulary=16
 6: (lo, w) -> low; count=7; vocabulary=17
 7: (new, er_) -> newer_; count=6; vocabulary=18
 8: (low, _) -> low_; count=5; vocabulary=19
 9: (d, er_) -> der_; count=3; vocabulary=20
10: (i, der_) -> ider_; count=3; vocabulary=21
11: (w, ider_) -> wider_; count=3; vocabulary=22
12: (e, s) -> es; count=2; vocabulary=23
13: (es, t) -> est; count=2; vocabulary=24
14: (est, _) -> est_; count=2; vocabulary=25
15: (low, est_) -> lowest_; count=2; vocabulary=26
16: (new, _) -> new_; count=2; vocabulary=27
new ['new_']
newer ['newer_']
lowest ['lowest_']
widest ['w', 'i', 'd', 'est_']
newestest ['new', 'est', 'est_']


In [38]:
import re

PARAGRAPH = (
    "Students study language because language connects people. "
    "A student studies words and learns how useful word parts repeat. "
    "Teachers explain tokenization, while students compare tokens "
    "from different sentences. Reusable patterns help readers "
    "understand unfamiliar words such as microarchitecture. "
    "Learning these patterns makes reading and writing easier."
)
PARAGRAPH_TARGETS = [
    "language", "students", "tokenization", "microarchitecture", "reusable"
]
words = re.findall(r"[a-z]+", PARAGRAPH.lower())
history, vocabulary, _ = train_bpe(words, 30)
print_history(history)
top_five = sorted(
    history, key=lambda x: (-x["count"], x["step"])
)[:5]
longest = sorted(
    vocabulary, key=lambda t: (-len(t.rstrip("_")), t)
)[:5]
print("Most frequent:", top_five)
print("Longest:", longest)
for word in PARAGRAPH_TARGETS:
    print(word, encode_bpe(word, history))

 1: (s, _) -> s_; count=16; vocabulary=25
 2: (e, _) -> e_; count=9; vocabulary=26
 3: (e, n) -> en; count=8; vocabulary=27
 4: (e, a) -> ea; count=7; vocabulary=28
 5: (e, r) -> er; count=7; vocabulary=29
 6: (s, t) -> st; count=6; vocabulary=30
 7: (a, n) -> an; count=5; vocabulary=31
 8: (en, t) -> ent; count=5; vocabulary=32
 9: (st, u) -> stu; count=5; vocabulary=33
10: (stu, d) -> stud; count=5; vocabulary=34
11: (a, r) -> ar; count=4; vocabulary=35
12: (d, _) -> d_; count=4; vocabulary=36
13: (i, n) -> in; count=4; vocabulary=37
14: (a, t) -> at; count=3; vocabulary=38
15: (an, d_) -> and_; count=3; vocabulary=39
16: (c, h) -> ch; count=3; vocabulary=40
17: (e, c) -> ec; count=3; vocabulary=41
18: (e, s_) -> es_; count=3; vocabulary=42
19: (g, _) -> g_; count=3; vocabulary=43
20: (in, g_) -> ing_; count=3; vocabulary=44
21: (l, e_) -> le_; count=3; vocabulary=45
22: (n, s_) -> ns_; count=3; vocabulary=46
23: (o, r) -> or; count=3; vocabulary=47
24: (stud, ent) -> student; count=

In [39]:
from nltk.tokenize import TreebankWordTokenizer, MWETokenizer

SENTENCES = [
    "The students' project is working well in New York.",
    "They're building a state-of-the-art app, and it doesn't "
    "crash during testing.",
    "The teachers' feedback helped the team break the ice.",
    "We enjoyed ice cream after finishing the app.",
]

# Manual policy: separate punctuation, clitics, and selected suffixes.
# Keep hyphenated compounds intact. MWEs are grouped separately below.
MANUAL_SENTENCES = [
    ["The", "student", "s", "'", "project", "is", "work", "ing",
     "well", "in", "New", "York", "."],
    ["They", "'re", "build", "ing", "a", "state-of-the-art",
     "app", ",", "and", "it", "does", "n't", "crash", "during",
     "test", "ing", "."],
    ["The", "teacher", "s", "'", "feedback", "help", "ed", "the",
     "team", "break", "the", "ice", "."],
    ["We", "enjoy", "ed", "ice", "cream", "after", "finish", "ing",
     "the", "app", "."],
]
MWES = [("New", "York"), ("break", "the", "ice"), ("ice", "cream")]

paragraph = " ".join(SENTENCES)
naive = paragraph.split()
manual = [t for s in MANUAL_SENTENCES for t in s]
tokenizer = TreebankWordTokenizer()
tool_sentences = [tokenizer.tokenize(s) for s in SENTENCES]
tool = [t for s in tool_sentences for t in s]
grouped = MWETokenizer(MWES, separator="_").tokenize(tool)
print("Naive:", naive)
print("Manual:", manual)
print("NLTK:", tool)
print("With MWEs:", grouped)

Naive: ['The', "students'", 'project', 'is', 'working', 'well', 'in', 'New', 'York.', "They're", 'building', 'a', 'state-of-the-art', 'app,', 'and', 'it', "doesn't", 'crash', 'during', 'testing.', 'The', "teachers'", 'feedback', 'helped', 'the', 'team', 'break', 'the', 'ice.', 'We', 'enjoyed', 'ice', 'cream', 'after', 'finishing', 'the', 'app.']
Manual: ['The', 'student', 's', "'", 'project', 'is', 'work', 'ing', 'well', 'in', 'New', 'York', '.', 'They', "'re", 'build', 'ing', 'a', 'state-of-the-art', 'app', ',', 'and', 'it', 'does', "n't", 'crash', 'during', 'test', 'ing', '.', 'The', 'teacher', 's', "'", 'feedback', 'help', 'ed', 'the', 'team', 'break', 'the', 'ice', '.', 'We', 'enjoy', 'ed', 'ice', 'cream', 'after', 'finish', 'ing', 'the', 'app', '.']
NLTK: ['The', 'students', "'", 'project', 'is', 'working', 'well', 'in', 'New', 'York', '.', 'They', "'re", 'building', 'a', 'state-of-the-art', 'app', ',', 'and', 'it', 'does', "n't", 'crash', 'during', 'testing', '.', 'The', 'teacher